In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

import re

In [3]:
df1 = pd.read_csv("../datasets/DailyDialog.csv")
df2 = pd.read_csv("../datasets/goemotions.csv")
df3 = pd.read_csv(
    "../datasets/twitter_training.csv",
    header=None,
    names=["id","entity","sentiment","text"]
)
df4 = pd.read_csv("../datasets/Mental Health dataset.csv")
df5 = pd.read_csv("../datasets/Counseling Conversations.csv")

In [4]:
print("DailyDialog")
print(df1.columns)
print()
print("GoEmotions")
print(df2.columns)
print()
print("Twitter")
print(df3.columns)
print()
print("Mental Health")
print(df4.columns)
print("Counseling")
print(df5.columns)

DailyDialog
Index(['text', 'sentiment'], dtype='object')

GoEmotions
Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='object')

Twitter
Index(['id', 'entity', 'sentiment', 'text'], dtype='object')

Mental Health
Index(['Unnamed: 0', 'statement', 'status'], dtype='object')
Counseling
Index(['Context', 'Response'], dtype='object')


In [5]:
df4 = df4.rename(columns={
    "statement": "text",
    "status": "sentiment"
})

df4 = df4[["text", "sentiment"]]

In [6]:
print(df4.head())

                                                text sentiment
0                                         oh my gosh   Anxiety
1  trouble sleeping, confused mind, restless hear...   Anxiety
2  All wrong, back off dear, forward doubt. Stay ...   Anxiety
3  I've shifted my focus to something else but I'...   Anxiety
4  I'm restless and restless, it's been a month n...   Anxiety


In [7]:
df4["sentiment"] = df4["sentiment"].str.lower()

mental_mapping = {
    "normal": "neutral",
    "anxiety": "fear",
    "stress": "nervousness",
    "depression": "sadness",
    "suicidal": "grief"
}

df4["sentiment"] = df4["sentiment"].replace(mental_mapping)

df4 = df4[~df4["sentiment"].isin([
    "bipolar",
    "personality disorder"
])]

In [8]:
df1 = df1[["text", "sentiment"]]

df3 = df3[["text", "sentiment"]]

df4 = df4[["text", "sentiment"]]
df4.columns = ["text", "sentiment"]

df5 = df5[["Context"]]
df5.columns = ["text"]
df5["sentiment"] = "neutral"

print(df1.head())
print(df3.head())
print(df4.head())
print(df5.head())

                                                text sentiment
0  I experienced this emotion when my grandfather...   sadness
1   when I first moved in , I walked everywhere ....   neutral
2  ` Oh ! " she bleated , her voice high and rath...     anger
3  However , does the right hon. Gentleman recogn...      fear
4  My boyfriend didn't turn up after promising th...   sadness
                                                text sentiment
0  im getting on borderlands and i will murder yo...  Positive
1  I am coming to the borders and I will kill you...  Positive
2  im getting on borderlands and i will kill you ...  Positive
3  im coming on borderlands and i will murder you...  Positive
4  im getting on borderlands 2 and i will murder ...  Positive
                                                text sentiment
0                                         oh my gosh      fear
1  trouble sleeping, confused mind, restless hear...      fear
2  All wrong, back off dear, forward doubt. Stay ...   

In [9]:
emotion_columns = [
    'admiration','amusement','anger','annoyance','approval',
    'caring','confusion','curiosity','desire','disappointment',
    'disapproval','disgust','embarrassment','excitement','fear',
    'gratitude','grief','joy','love','nervousness','optimism',
    'pride','realization','relief','remorse','sadness',
    'surprise','neutral'
]

df2["sentiment"] = df2[emotion_columns].idxmax(axis=1)

df2 = df2[["text","sentiment"]]

print(df2.head())

                                                text   sentiment
0                                    That game hurt.     sadness
1   >sexuality shouldn’t be a grouping category I...  admiration
2     You do right, if you don't care then fuck 'em!     neutral
3                                 Man I love reddit.        love
4  [NAME] was nowhere near them, he was by the Fa...     neutral


In [10]:
df3["sentiment"] = df3["sentiment"].str.lower()
df4["sentiment"] = df4["sentiment"].str.lower()
df5["sentiment"] = df5["sentiment"].str.lower()

In [11]:
print(df4["sentiment"].unique())

['fear' 'neutral' 'sadness' 'grief' nan 'nervousness']


In [12]:
twitter_mapping = {
    "positive": "joy",
    "negative": "sadness",
    "neutral": "neutral",
    "irrelevant": "neutral"
}

df3["sentiment"] = df3["sentiment"].str.lower()
df3["sentiment"] = df3["sentiment"].replace(twitter_mapping)

In [13]:
df1 = df1[["text", "sentiment"]]
df2 = df2[["text", "sentiment"]]
df3 = df3[["text", "sentiment"]]

In [14]:
emotion_df = pd.concat([df1, df2, df4], ignore_index=True)

In [15]:
emotion_df = emotion_df.dropna()
emotion_df = emotion_df.drop_duplicates()
emotion_df = emotion_df.reset_index(drop=True)

In [16]:
print(emotion_df.shape)
print(emotion_df["sentiment"].value_counts())

(200235, 2)
sentiment
neutral           48949
sadness           20281
admiration        12377
approval          11882
grief             11053
annoyance          9128
anger              7761
fear               7493
disapproval        6829
joy                6219
disappointment     5492
curiosity          5204
amusement          5015
confusion          4816
gratitude          4441
realization        4386
optimism           3837
caring             3809
excitement         3441
nervousness        3120
love               3117
disgust            2728
surprise           2604
desire             2286
embarrassment      1415
remorse            1197
relief              706
pride               649
Name: count, dtype: int64


In [17]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


[nltk_data] Downloading package stopwords to C:\Users\Pravart
[nltk_data]     singh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Pravart
[nltk_data]     singh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Pravart
[nltk_data]     singh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [18]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# keep important negation words
negation_words = {
    "not",
    "no",
    "nor",
    "never"
}

# remove negations from stopwords
stop_words = stop_words - negation_words


def clean_text(text):
    text = str(text).lower()

    # expand contractions
    text = text.replace("n't", " not")

    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)

    text = re.sub(r"\d+", "", text)

    text = text.translate(str.maketrans("", "", string.punctuation))

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    if len(words) == 0:
        return "empty"

    return " ".join(words)

In [19]:
test_sentences = [
    "I am happy",
    "I am not happy",
    "I never loved him",
    "I don't like this"
]

for sentence in test_sentences:
    print(sentence, "---->", clean_text(sentence))

I am happy ----> happy
I am not happy ----> not happy
I never loved him ----> never loved
I don't like this ----> not like


In [20]:
emotion_df["clean_text"] = emotion_df["text"].apply(clean_text)

In [21]:
print(emotion_df["clean_text"].isna().sum())

0


In [22]:
print((emotion_df["clean_text"] == "").sum())

0


In [23]:
emotion_df["clean_text"] = emotion_df["clean_text"].replace("", "empty")

In [24]:
emotion_df[["text", "clean_text", "sentiment"]].head(10)

,text,clean_text,sentiment
0,I experienced this emotion when my grandfather...,experienced emotion grandfather passed away,sadness
1,"when I first moved in , I walked everywhere ....",first moved walked everywhere within week purs...,neutral
2,"` Oh ! "" she bleated , her voice high and rath...",oh bleated voice high rather indignant,anger
3,"However , does the right hon. Gentleman recogn...",however right hon gentleman recognise profound...,fear
4,My boyfriend didn't turn up after promising th...,boyfriend not turn promising coming,sadness
5,It's freezing .,freezing,neutral
6,That ’ s not all ! I also had to finish writi...,’ not also finish writing sale report bos end ...,sadness
7,I don't have a warrant .,not warrant,anger
8,I guess so .,guess,neutral
9,I was just robbed !,robbed,sadness


In [25]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

emotion_df["label"] = label_encoder.fit_transform(emotion_df["sentiment"])

print(emotion_df.head())
print(emotion_df["label"].nunique())

                                                text sentiment  \
0  I experienced this emotion when my grandfather...   sadness   
1   when I first moved in , I walked everywhere ....   neutral   
2  ` Oh ! " she bleated , her voice high and rath...     anger   
3  However , does the right hon. Gentleman recogn...      fear   
4  My boyfriend didn't turn up after promising th...   sadness   

                                          clean_text  label  
0        experienced emotion grandfather passed away     26  
1  first moved walked everywhere within week purs...     20  
2             oh bleated voice high rather indignant      2  
3  however right hon gentleman recognise profound...     14  
4                boyfriend not turn promising coming     26  
28


In [26]:
main_emotion_mapping = {

    # Happy
    "joy": "Happy",
    "admiration": "Happy",
    "gratitude": "Happy",
    "love": "Happy",
    "optimism": "Happy",
    "pride": "Happy",
    "amusement": "Happy",
    "excitement": "Happy",

    # Sad
    "sadness": "Sad",
    "grief": "Sad",
    "remorse": "Sad",
    "disappointment": "Sad",

    # Angry
    "anger": "Angry",
    "annoyance": "Angry",
    "disgust": "Angry",
    "disapproval": "Angry",

    # Fear
    "fear": "Fear",
    "nervousness": "Fear",
    "confusion": "Fear",

    # Neutral
    "neutral": "Neutral",
    "realization": "Neutral",

    # Affection
    "approval": "Affection",
    "caring": "Affection",

    # Curiosity
    "curiosity": "Curiosity",
    "desire": "Curiosity",
    "surprise": "Curiosity",

    # Relief
    "relief": "Relief",

    # Embarrassment
    "embarrassment": "Embarrassment"
}

emotion_df["main_emotion"] = emotion_df["sentiment"].map(main_emotion_mapping)

In [27]:
main_encoder = LabelEncoder()

emotion_df["main_label"] = main_encoder.fit_transform(
    emotion_df["main_emotion"]
)

In [28]:
print(emotion_df[["sentiment","main_emotion"]].head(20))
print()
print(emotion_df["main_emotion"].value_counts())
print()
print(emotion_df["main_emotion"].nunique())

   sentiment main_emotion
0    sadness          Sad
1    neutral      Neutral
2      anger        Angry
3       fear         Fear
4    sadness          Sad
5    neutral      Neutral
6    sadness          Sad
7      anger        Angry
8    neutral      Neutral
9    sadness          Sad
10   neutral      Neutral
11   neutral      Neutral
12      fear         Fear
13     anger        Angry
14   neutral      Neutral
15   sadness          Sad
16   sadness          Sad
17   neutral      Neutral
18       joy        Happy
19      fear         Fear

main_emotion
Neutral          53335
Happy            39096
Sad              38023
Angry            26446
Affection        15691
Fear             15429
Curiosity        10094
Embarrassment     1415
Relief             706
Name: count, dtype: int64

9


In [29]:
emotion_df.to_csv("final_emotion_dataset.csv", index=False)

In [30]:
print(emotion_df.shape)
emotion_df.head()

(200235, 6)


,text,sentiment,clean_text,label,main_emotion,main_label
0,I experienced this emotion when my grandfather...,sadness,experienced emotion grandfather passed away,26,Sad,8
1,"when I first moved in , I walked everywhere ....",neutral,first moved walked everywhere within week purs...,20,Neutral,6
2,"` Oh ! "" she bleated , her voice high and rath...",anger,oh bleated voice high rather indignant,2,Angry,1
3,"However , does the right hon. Gentleman recogn...",fear,however right hon gentleman recognise profound...,14,Fear,4
4,My boyfriend didn't turn up after promising th...,sadness,boyfriend not turn promising coming,26,Sad,8


In [31]:
print(emotion_df["sentiment"].value_counts())
print("Total emotions:", emotion_df["sentiment"].nunique())

sentiment
neutral           48949
sadness           20281
admiration        12377
approval          11882
grief             11053
annoyance          9128
anger              7761
fear               7493
disapproval        6829
joy                6219
disappointment     5492
curiosity          5204
amusement          5015
confusion          4816
gratitude          4441
realization        4386
optimism           3837
caring             3809
excitement         3441
nervousness        3120
love               3117
disgust            2728
surprise           2604
desire             2286
embarrassment      1415
remorse            1197
relief              706
pride               649
Name: count, dtype: int64
Total emotions: 28


In [32]:
print(df2.columns)

Index(['text', 'sentiment'], dtype='object')


In [33]:
print(emotion_df["main_emotion"].value_counts())
print(emotion_df["main_emotion"].nunique())

main_emotion
Neutral          53335
Happy            39096
Sad              38023
Angry            26446
Affection        15691
Fear             15429
Curiosity        10094
Embarrassment     1415
Relief             706
Name: count, dtype: int64
9
